### Structure prediction and self-consistency validation nwith ESMFold

#### Objective
Since ProteinMPNN cannot tell, whether the sequence has a single dominant fold, or whether the fold is the intended one, for each sequence generate an independent structure using ESMfold and measure how the prediction returns to the backbone sequence. Afterwards, select a candidate set for complex prediction.

#### Methodology
ESMFold was chosen because it can predict the structure from a single sequence with no homologous proteins. <br>
Then the self consistency test is recorded based on following metrics: <br>
1. pLDDT (predictor's own per residue confidence)
2. scRMSD (whole-chain CA agreement with target backbone)
3. Motif RMSD (gated at >90, <1.5 Å and <1.0 Å ) <br>


Additionally, ESM-2 pseudo-log-likelihood is computed for all 290 designs
against four natural references, Neo-2/15, and a composition-matched scrambled floor.

#### Limitaions
1. No information about thermodynamic stability, expression, binding and solubility.
2. Self-consistency is not implying the active functionality

In [ ]:
# Load generated in stage 3 sequences and survivor tables in stage 2.
import os, json, glob, re, time
import numpy as np, pandas as pd

from google.colab import drive
drive.mount("/content/drive")

PROJ = "/content/drive/MyDrive/il2_mimetic"
OUT2 = f"{PROJ}/stage2_backbones"
OUT3 = f"{PROJ}/stage3_seqs"
OUT4 = f"{PROJ}/stage4_fold"
os.makedirs(f"{OUT4}/pdb", exist_ok=True)

with open(f"{PROJ}/stage1_motif/params.json") as fh:
    params = json.load(fh)
CHAIN = params["chains"]

survivors = pd.read_csv(f"{OUT2}/survivors.tsv", sep="\t")
df3       = pd.read_csv(f"{OUT3}/sequences.tsv", sep="\t")

print(f"{len(survivors)} backbones, {len(df3)} sequences")
print(df3.groupby('arm').size().to_string())

Mounted at /content/drive
29 backbones, 290 sequences
arm
fixAll         145
fixContacts    145


In [ ]:
# transformers carries the ESMFold implementation
import subprocess
subprocess.run(["pip", "-q", "install", "biopython", "transformers"], check=True)

CompletedProcess(args=['pip', '-q', 'install', 'biopython', 'transformers'], returncode=0)

In [ ]:
# Load ESMFold and fold one sequence to inspect the output structure
# before committing to the entire set
import torch
from transformers import AutoTokenizer, EsmForProteinFolding

OUT4 = f"{PROJ}/stage4_fold"
os.makedirs(f"{OUT4}/pdb", exist_ok=True)

tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained(
    "facebook/esmfold_v1", low_cpu_mem_usage=True)

model = model.cuda()
model.esm = model.esm.half()          # halves memory, negligible accuracy cost
model.trunk.set_chunk_size(64)        # keeps peak VRAM down on an L4
torch.backends.cuda.matmul.allow_tf32 = True
model.eval()
print("loaded |", torch.cuda.get_device_name(0))

# --- fold one sequence and inspect what comes back ---
import time
test = df3.iloc[0]
print(f"\ntest: {test.seq_id} ({test.length} aa)")

t0 = time.time()
inp = tok([test.sequence], return_tensors="pt",
          add_special_tokens=False).to("cuda")
with torch.no_grad():
    out = model(**inp)
dt = time.time() - t0

print(f"folded in {dt:.1f} s")
print("output keys:", [k for k in out.keys()])
print("plddt shape:", out["plddt"].shape)
print("positions   :", out["positions"].shape)

pdb_str = model.output_to_pdb(out)[0]
print("\npdb lines:", len(pdb_str.splitlines()))
print(pdb_str.splitlines()[0][:70])

config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 8.44GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 8.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/4498 [00:00<?, ?it/s]

[transformers] EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status  | 
-----------------------------------+---------+-
esm.contact_head.regression.bias   | MISSING | 
esm.contact_head.regression.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


loaded | NVIDIA L4

test: pad0_len100_0_fixAll_0 (96 aa)
folded in 5.0 s
output keys: ['frames', 'sidechain_frames', 'unnormalized_angles', 'angles', 'positions', 'states', 's_s', 's_z', 'distogram_logits', 'lm_logits', 'aatype', 'atom14_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'atom37_atom_exists', 'residue_index', 'lddt_head', 'plddt', 'ptm_logits', 'ptm', 'aligned_confidence_probs', 'predicted_aligned_error', 'max_predicted_aligned_error']
plddt shape: torch.Size([1, 96, 37])
positions   : torch.Size([8, 1, 96, 14, 3])

pdb lines: 797
PARENT N/A


In [ ]:
# Fold all 290 and measure self-consistency against the RFdiffusion
# backbone each sequence was designed for. pLDDT is taken from the CA
# column of the per-atom tensor. Progress is checkpointed every 25
# structures so that a disconnected session resumes rather than restarts it
from Bio.PDB import PDBParser, Superimposer
from Bio.PDB.Polypeptide import is_aa

parser = PDBParser(QUIET=True)
METRICS = f"{OUT4}/fold_metrics.tsv"

def ca_atoms(chain, positions=None):
    out = []
    for r in chain:
        if not is_aa(r, standard=True) or "CA" not in r:
            continue
        if positions is None or r.id[1] in positions:
            out.append(r["CA"])
    return out

# motif positions in design numbering, per backbone
motif_pos = {}
for _, r in survivors.iterrows():
    trb = np.load(f"{OUT2}/{r.variant}/{r.design}.trb", allow_pickle=True)
    motif_pos[r.design] = set(int(h) for c, h in trb["con_hal_pdb_idx"])

done = {}
if os.path.exists(METRICS):
    prev = pd.read_csv(METRICS, sep="\t")
    done = {r.seq_id: r for _, r in prev.iterrows()}
    print(f"resuming — {len(done)} already folded")

rows, t_all = list(done.values()), time.time()
for i, (_, s) in enumerate(df3.iterrows(), 1):
    if s.seq_id in done:
        continue

    inp = tok([s.sequence], return_tensors="pt",
              add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model(**inp)

    pl = out["plddt"][0, :, 1].detach().cpu().numpy()
    if pl.max() <= 1.0:
        pl = pl * 100

    pred_path = f"{OUT4}/pdb/{s.seq_id}.pdb"
    with open(pred_path, "w") as fh:
        fh.write(model.output_to_pdb(out)[0])

    # superpose onto the RFdiffusion backbone it was designed for
    bb   = parser.get_structure("b", f"{OUT2}/{s.variant}/{s.backbone}.pdb")[0]["A"]
    pred = parser.get_structure("p", pred_path)[0]["A"]

    sup = Superimposer()
    sup.set_atoms(ca_atoms(bb), ca_atoms(pred))
    sc_rmsd = sup.rms

    mp = motif_pos[s.backbone]
    sup_m = Superimposer()
    sup_m.set_atoms(ca_atoms(bb, mp), ca_atoms(pred, mp))
    motif_rmsd = sup_m.rms

    rows.append({"seq_id": s.seq_id, "arm": s.arm, "variant": s.variant,
                 "backbone": s.backbone, "length": s.length,
                 "mpnn_score": s.mpnn_score,
                 "plddt": round(float(pl.mean()), 2),
                 "sc_rmsd": round(float(sc_rmsd), 3),
                 "motif_rmsd": round(float(motif_rmsd), 3)})

    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_csv(METRICS, sep="\t", index=False)
        el = time.time() - t_all
        print(f"  {len(rows)}/{len(df3)}  "
              f"({el/60:.1f} min, ~{(len(df3)-len(rows))*el/max(1,i)/60:.0f} min left)",
              flush=True)
        torch.cuda.empty_cache()

df4 = pd.DataFrame(rows)
df4.to_csv(METRICS, sep="\t", index=False)
print(f"\nfolded {len(df4)} sequences in {(time.time()-t_all)/60:.1f} min")

  25/290  (0.8 min, ~8 min left)
  50/290  (1.5 min, ~7 min left)
  75/290  (2.3 min, ~7 min left)
  100/290  (3.0 min, ~6 min left)
  125/290  (3.8 min, ~5 min left)
  150/290  (4.6 min, ~4 min left)
  175/290  (5.3 min, ~3 min left)
  200/290  (6.0 min, ~3 min left)
  225/290  (6.7 min, ~2 min left)
  250/290  (7.4 min, ~1 min left)
  275/290  (8.1 min, ~0 min left)

folded 290 sequences in 8.5 min


In [ ]:
# Apply the pre-registered gates, then ask whether ProteinMPNN score
# predicted any of this. The pooled correlation is reported here and
# decomposed in the next cell, because the two arms differ in both
# MPNN score and pLDDT and pooling them would manufacture a correlation
# from that difference alone
PLDDT_GATE, SCRMSD_GATE, MOTIF_GATE = 90.0, 1.5, 1.0   # pre-registered

df4["pass_plddt"] = df4.plddt      > PLDDT_GATE
df4["pass_scrmsd"] = df4.sc_rmsd   < SCRMSD_GATE
df4["pass_motif"]  = df4.motif_rmsd < MOTIF_GATE
df4["pass"] = df4.pass_plddt & df4.pass_scrmsd & df4.pass_motif
df4.to_csv(METRICS, sep="\t", index=False)

print("Stage 4 funnel")
for c, label in [("pass_plddt", f"pLDDT > {PLDDT_GATE}"),
                 ("pass_scrmsd", f"scRMSD < {SCRMSD_GATE} Å"),
                 ("pass_motif",  f"motif RMSD < {MOTIF_GATE} Å"),
                 ("pass", "all three")]:
    print(f"  {label:22s} {df4[c].sum():3d}/{len(df4)}  ({100*df4[c].mean():.0f}%)")

print("\n--- does MPNN score predict folding? ---")
print(df4.groupby("arm").agg(
    n=("seq_id", "count"),
    mpnn=("mpnn_score", "mean"),
    plddt=("plddt", "mean"),
    scrmsd=("sc_rmsd", "median"),
    passed=("pass", "sum")).round(3).to_string())

print("\ncorrelation mpnn_score vs plddt :",
      round(df4.mpnn_score.corr(df4.plddt), 3))
print("correlation mpnn_score vs scRMSD:",
      round(df4.mpnn_score.corr(df4.sc_rmsd), 3))

Stage 4 funnel
  pLDDT > 90.0           118/290  (41%)
  scRMSD < 1.5 Å         222/290  (77%)
  motif RMSD < 1.0 Å     236/290  (81%)
  all three              115/290  (40%)

--- does MPNN score predict folding? ---
               n   mpnn   plddt  scrmsd  passed
arm                                            
fixAll       145  1.291  83.532   0.890      50
fixContacts  145  1.196  87.948   0.824      65

correlation mpnn_score vs plddt : -0.256
correlation mpnn_score vs scRMSD: 0.087


In [ ]:
# Within-arm correlations and pass rate by MPNN quartile. If MPNN score
# carried information about foldability, the best quartile should pass
# more often than the worst within each arm
print("within-arm correlations:")
for arm, g in df4.groupby("arm"):
    print(f"  {arm:12s} n={len(g):3d}  "
          f"mpnn~plddt {g.mpnn_score.corr(g.plddt):+.3f}  "
          f"mpnn~scRMSD {g.mpnn_score.corr(g.sc_rmsd):+.3f}")

print("\nper-sequence pass rate by MPNN score quartile, within arm:")
df4["q"] = df4.groupby("arm").mpnn_score.transform(
    lambda s: pd.qcut(s, 4, labels=["best", "2nd", "3rd", "worst"]))
print(df4.groupby(["arm", "q"], observed=True)["pass"]
         .agg(["count", "sum", "mean"]).round(2).to_string())

within-arm correlations:
  fixAll       n=145  mpnn~plddt -0.173  mpnn~scRMSD +0.062
  fixContacts  n=145  mpnn~plddt -0.018  mpnn~scRMSD +0.028

per-sequence pass rate by MPNN score quartile, within arm:
                   count  sum  mean
arm         q                      
fixAll      best      37    9  0.24
            2nd       36   16  0.44
            3rd       36   15  0.42
            worst     36   10  0.28
fixContacts best      37   18  0.49
            2nd       36   11  0.31
            3rd       36   17  0.47
            worst     36   19  0.53


In [ ]:
# Formal test of the quartile pattern, so the absence of a trend is a
# measured null rather than an impression from four percentages
from scipy.stats import chi2_contingency
for arm, g in df4.groupby("arm"):
    tab = pd.crosstab(g.q, g["pass"])
    chi2, p, _, _ = chi2_contingency(tab)
    print(f"{arm:12s} chi2={chi2:.2f}  p={p:.3f}")

fixAll       chi2=4.81  p=0.186
fixContacts  chi2=4.19  p=0.242


In [ ]:
# Rank survivors by an equally weighted z-score composite, then take the
# best sequence per backbone before taking the top 12. Selecting per
# backbone first prevents one well-behaved backbone from occupying the
# entire Stage 5 with near-identical sequences
ok = df4[df4["pass"]].copy()
print(f"{len(ok)} survivors from {ok.backbone.nunique()} backbones")

# composite: normalise each metric, lower is better
for c, invert in [("plddt", True), ("sc_rmsd", False), ("motif_rmsd", False)]:
    v = ok[c]
    z = (v - v.mean()) / v.std()
    ok[f"z_{c}"] = -z if invert else z
ok["composite"] = ok[["z_plddt", "z_sc_rmsd", "z_motif_rmsd"]].mean(axis=1)

# one sequence per backbone, then the best backbones
best = (ok.sort_values("composite")
          .groupby("backbone", as_index=False).first()
          .sort_values("composite"))

N_STAGE5 = 12
picked = best.head(N_STAGE5)

print(f"\nselected {len(picked)} from {len(best)} backbones")
print(picked[["seq_id", "arm", "variant", "length",
              "plddt", "sc_rmsd", "motif_rmsd", "composite"]]
      .round(3).to_string(index=False))
print("\nby arm   :", picked.arm.value_counts().to_dict())
print("by variant:", picked.variant.value_counts().to_dict())

picked.to_csv(f"{OUT4}/stage5_candidates.tsv", sep="\t", index=False)

115 survivors from 20 backbones

selected 12 from 20 backbones
                      seq_id         arm     variant  length  plddt  sc_rmsd  motif_rmsd  composite
 pad0_len100_0_fixContacts_1 fixContacts pad0_len100      96  93.57    0.405       0.325     -1.636
     pad0_len100_15_fixAll_4      fixAll pad0_len100     105  92.46    0.619       0.300     -0.883
pad2_len100_28_fixContacts_3 fixContacts pad2_len100     108  94.65    0.742       0.417     -0.868
     pad0_len100_14_fixAll_1      fixAll pad0_len100     113  91.47    0.516       0.292     -0.839
 pad0_len100_2_fixContacts_3 fixContacts pad0_len100     115  92.01    0.552       0.317     -0.836
      pad2_len100_7_fixAll_3      fixAll pad2_len100      91  92.84    0.647       0.353     -0.748
pad2_len100_27_fixContacts_0 fixContacts pad2_len100     102  91.84    0.581       0.327     -0.680
     pad0_len100_22_fixAll_0      fixAll pad0_len100     107  92.76    0.624       0.395     -0.626
pad0_len100_37_fixContacts_2 fixConta

In [ ]:
# Release ESMFold before loading ESM-2: both exceed half the available
# VRAM and will not coexist
del model
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated()/1e9:.1f} GB still allocated")

0.0 GB still allocated


In [ ]:
# Persist gates, per-arm summaries and the MPNN correlations, so the
# selection criteria are recoverable without re-reading the notebook
import json

with open(f"{OUT4}/params.json", "w") as fh:
    json.dump({
        "model": "facebook/esmfold_v1",
        "precision": "esm half, trunk chunk_size 64",
        "gates": {"plddt": 90.0, "sc_rmsd": 1.5, "motif_rmsd": 1.0},
        "n_sequences": int(len(df4)),
        "n_passed": int(df4["pass"].sum()),
        "n_backbones_represented": int(df4[df4["pass"]].backbone.nunique()),
        "n_selected_stage5": 12,
        "by_arm": {k: {"plddt": round(float(g.plddt.mean()), 2),
                       "sc_rmsd": round(float(g.sc_rmsd.median()), 3),
                       "passed": int(g["pass"].sum()), "n": int(len(g))}
                   for k, g in df4.groupby("arm")},
        "mpnn_within_arm_corr": {
            k: {"plddt": round(float(g.mpnn_score.corr(g.plddt)), 3),
                "sc_rmsd": round(float(g.mpnn_score.corr(g.sc_rmsd)), 3)}
            for k, g in df4.groupby("arm")},
    }, fh, indent=2)

df4.to_csv(f"{OUT4}/fold_metrics.tsv", sep="\t", index=False)
print("saved:")
for f in sorted(os.listdir(OUT4)):
    print("  ", f)
print(f"\n{len(os.listdir(f'{OUT4}/pdb'))} predicted structures")

saved:
   fold_metrics.tsv
   params.json
   pdb
   stage5_candidates.tsv

290 predicted structures


Can a sequence-only model substitute for folding?

In [ ]:
# Load ESM-2 650M and define pseudo-log-likelihood: mask each position in
# turn and score the true residue. Averaging per residue makes the metric
# length-independent, which is required to compare a 100-residue design
# against a 551-residue receptor chain
import torch, gc, random
from transformers import AutoTokenizer, AutoModelForMaskedLM
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

# free ESMFold if still loaded
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

esm_tok = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
esm = AutoModelForMaskedLM.from_pretrained(
    "facebook/esm2_t33_650M_UR50D", torch_dtype=torch.float16).cuda().eval()
print("ESM-2 650M loaded")

@torch.no_grad()
def pll(seq, chunk=64):
    """Mean per-residue pseudo-log-likelihood: mask each position, score the
    true residue. Length-independent; higher = more 'natural' to the model."""
    ids = esm_tok(seq, return_tensors="pt").input_ids.cuda()
    L   = ids.shape[1] - 2                       # strip BOS/EOS
    total = 0.0
    for s in range(0, L, chunk):
        idx   = list(range(s, min(s + chunk, L)))
        batch = ids.repeat(len(idx), 1)
        for row, p in enumerate(idx):
            batch[row, p + 1] = esm_tok.mask_token_id
        lp = torch.log_softmax(esm(batch).logits.float(), dim=-1)
        for row, p in enumerate(idx):
            total += lp[row, p + 1, ids[0, p + 1]].item()
    return total / L

# reference sequences
refs = {}
st = PDBParser(QUIET=True).get_structure("n", f"{PROJ}/2B5I.pdb")[0]
for cid, label in {v: k for k, v in CHAIN.items()}.items():
    refs[f"natural:{label}"] = "".join(
        seq1(r.get_resname()) for r in st[cid] if is_aa(r, standard=True))

neo_pdb = f"{OUT3}/benchmark/6DG6.pdb"
if os.path.exists(neo_pdb):
    nst = PDBParser(QUIET=True).get_structure("neo", neo_pdb)[0]["A"]
    refs["denovo:Neo-2/15"] = "".join(
        seq1(r.get_resname()) for r in nst if is_aa(r, standard=True))
else:
    print("WARNING: 6DG6 not found — Neo-2/15 reference will be missing")

for k, v in refs.items():
    print(f"  {k:22s} {len(v)} aa")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

ESM-2 650M loaded
  natural:IL2            120 aa
  natural:IL2Rb          196 aa
  natural:gc             191 aa
  natural:IL2Ra          122 aa
  denovo:Neo-2/15        100 aa


In [ ]:
# Score the designs against three reference classes: natural receptor
# chains, a published de novo design that is known to work, and
# composition-matched scrambled sequences as a floor. The floor is the
# part that matters — a metric that cannot beat shuffled sequence is not
# measuring design quality
random.seed(0)
rows = []

# references
for label, s in refs.items():
    grp = label.split(":")[0]
    rows.append({"group": grp, "name": label.split(":")[1],
                 "length": len(s), "pll": pll(s)})

# scrambled floor
for i, s in enumerate(df4.sample(20, random_state=0).seq_id):
    seq = df3.loc[df3.seq_id == s, "sequence"].iloc[0]
    sh  = "".join(random.sample(seq, len(seq)))
    rows.append({"group": "scrambled", "name": f"shuf_{i}",
                 "length": len(sh), "pll": pll(sh)})

# all designs
for i, (_, r) in enumerate(df3.iterrows(), 1):
    rows.append({"group": f"design:{r.arm}", "name": r.seq_id,
                 "length": r.length, "pll": pll(r.sequence)})
    if i % 50 == 0:
        print(f"  {i}/{len(df3)}", flush=True)

esm_df = pd.DataFrame(rows)
esm_df.to_csv(f"{OUT4}/esm2_pll.tsv", sep="\t", index=False)

print("\n--- mean per-residue pseudo-log-likelihood (higher = more natural) ---")
print(esm_df.groupby("group").pll.agg(["count", "mean", "std"])
          .round(3).to_string())
print("\nindividual references:")
print(esm_df[esm_df.group.isin(["natural", "denovo"])]
      [["group", "name", "pll"]].round(3).to_string(index=False))

  50/290
  100/290
  150/290
  200/290
  250/290

--- mean per-residue pseudo-log-likelihood (higher = more natural) ---
                    count   mean    std
group                                  
denovo                  1 -2.757    NaN
design:fixAll         145 -2.237  0.173
design:fixContacts    145 -2.071  0.152
natural                 4 -2.068  0.601
scrambled              20 -2.677  0.125

individual references:
  group     name    pll
natural      IL2 -2.834
natural    IL2Rb -2.246
natural       gc -1.497
natural    IL2Ra -1.696
 denovo Neo-2/15 -2.757


In [ ]:
# The PDB-derived sequences have unresolved loops spliced out, creating
# artificial junctions that depress PLL. Re-scoring the full-length
# UniProt sequences establishes how much of the natural references'
# poor showing is an artefact of that splicing
import urllib.request

UNIPROT = {"IL2": "P60568", "IL2Ra": "P01589",
           "IL2Rb": "P14784", "gc": "P31785"}
print("full UniProt sequences (no gaps):")
for name, acc in UNIPROT.items():
    txt = urllib.request.urlopen(
        f"https://rest.uniprot.org/uniprotkb/{acc}.fasta").read().decode()
    full = "".join(txt.split("\n")[1:])
    v = pll(full)
    gapped = esm_df.loc[esm_df.name == name, "pll"]
    print(f"  {name:6s} full {len(full):4d} aa  PLL {v:+.3f}   "
          f"(PDB-derived: {gapped.iloc[0]:+.3f})" if len(gapped) else "")

full UniProt sequences (no gaps):
  IL2    full  153 aa  PLL -2.568   (PDB-derived: -2.834)
  IL2Ra  full  272 aa  PLL -1.846   (PDB-derived: -1.696)
  IL2Rb  full  551 aa  PLL -2.226   (PDB-derived: -2.246)
  gc     full  369 aa  PLL -1.660   (PDB-derived: -1.497)


In [ ]:
# The same splicing concern applied to Neo-2/15 would invalidate the key
# comparison, so its numbering is checked for gaps. There are none, and
# its score stands as measured
nst = PDBParser(QUIET=True).get_structure("neo", f"{OUT3}/benchmark/6DG6.pdb")[0]["A"]
nums = sorted(r.id[1] for r in nst if is_aa(r, standard=True))
gaps = [(nums[i], nums[i+1]) for i in range(len(nums)-1) if nums[i+1]-nums[i] > 1]
print(f"Neo-2/15: {len(nums)} residues, numbering {nums[0]}-{nums[-1]}")
print("internal gaps:", gaps if gaps else "none — score is clean")

Neo-2/15: 100 residues, numbering 1-100
internal gaps: none — score is clean


In [ ]:
# Does PLL predict the structural outcome? Reported pooled and within
# arm, since the arms differ in both quantities and the pooled value
# inherits that difference
m = esm_df[esm_df.group.str.startswith("design")].merge(
        df4[["seq_id", "plddt", "sc_rmsd", "pass"]],
        left_on="name", right_on="seq_id")

print("ESM-2 PLL vs structural outcome:")
print(f"  corr with pLDDT  : {m.pll.corr(m.plddt):+.3f}")
print(f"  corr with scRMSD : {m.pll.corr(m.sc_rmsd):+.3f}")
print("\nwithin arm:")
for arm, g in m.groupby("group"):
    print(f"  {arm:20s} n={len(g):3d}  pll~plddt {g.pll.corr(g.plddt):+.3f}")
print("\nPLL by pass/fail:")
print(m.groupby("pass").pll.agg(["count", "mean", "std"]).round(3).to_string())

with open(f"{OUT4}/esm2_findings.json", "w") as fh:
    json.dump({
        "model": "facebook/esm2_t33_650M_UR50D",
        "metric": "mean per-residue pseudo-log-likelihood",
        "neo_2_15": -2.757,
        "neo_2_15_gaps": "none (6DG6 chain A, residues 1-100)",
        "scrambled_mean": -2.677,
        "native_IL2_fulllength": -2.568,
        "natural_mean_fulllength": -2.075,
        "design_fixContacts": -2.071,
        "design_fixAll": -2.237,
        "pll_vs_plddt_corr": round(float(m.pll.corr(m.plddt)), 3),
        "pll_vs_scrmsd_corr": round(float(m.pll.corr(m.sc_rmsd)), 3),
        "note": "PDB-derived natural sequences have spliced loops; "
                "full-length UniProt used for the reported natural values. "
                "Neo-2/15 structure is gap-free so its score is unaffected.",
    }, fh, indent=2)
print("\nsaved esm2_findings.json")

ESM-2 PLL vs structural outcome:
  corr with pLDDT  : +0.498
  corr with scRMSD : -0.146

within arm:
  design:fixAll        n=145  pll~plddt +0.548
  design:fixContacts   n=145  pll~plddt +0.227

PLL by pass/fail:
       count   mean    std
pass                      
False    175 -2.210  0.181
True     115 -2.069  0.150

saved esm2_findings.json


In [ ]:
# scRMSD is strongly right-skewed: median 0.86 A against a maximum of
# 15.5 A. The mean is not a usable summary and the median and IQR are
# reported instead
print(m.sc_rmsd.describe().round(3).to_string())
print("\nIQR:", round(m.sc_rmsd.quantile(.75) - m.sc_rmsd.quantile(.25), 3))

count    290.000
mean       1.407
std        1.753
min        0.397
25%        0.666
50%        0.863
75%        1.361
max       15.451

IQR: 0.695


In [ ]:
# Spearman over Pearson, because the scRMSD outliers dominate a
# product-moment correlation. Catastrophic failures are counted by arm,
# which turns out to be the clearest difference between them
from scipy.stats import spearmanr

r_p, p_p = spearmanr(m.pll, m.plddt)
r_s, p_s = spearmanr(m.pll, m.sc_rmsd)
print(f"Spearman  PLL ~ pLDDT  : {r_p:+.3f}  (p={p_p:.2g})")
print(f"Spearman  PLL ~ scRMSD : {r_s:+.3f}  (p={p_s:.2g})")

print(f"\ncatastrophic failures (scRMSD > 3 Å): {(m.sc_rmsd > 3).sum()}/{len(m)}")
print(m[m.sc_rmsd > 3].groupby("group").size().to_string())

Spearman  PLL ~ pLDDT  : +0.446  (p=1.5e-15)
Spearman  PLL ~ scRMSD : -0.283  (p=9.7e-07)

catastrophic failures (scRMSD > 3 Å): 24/290
group
design:fixAll         20
design:fixContacts     4


#### Results
| gate | pass |
|---|---|
| pLDDT > 90 | 118/290 (41%) |
| scRMSD < 1.5 Å | 222/290 (77%) |
| motif RMSD < 1.0 Å | 236/290 (81%) |
| all three | 115/290 (40%) |

1. Of the 118 designs clearing pLDDT, 115 clear everything. The other two gates remove three designs between them. pLDDT accounts for 97% of the surviving set. The RMSD gates are reported as near-redundant rather than as independent filters, which is not what was expected when they were pre-registered.

2. The two arms fail differently, not at different rates.
| arm | MPNN score | mean pLDDT | median scRMSD | passed | scRMSD > 3 Å |
|---|---|---|---|---|---|
| fixAll | 1.291 | 83.5 | 0.890 | 50/145 | **20** |
| fixContacts | 1.196 | 87.9 | 0.824 | 65/145 | **4** |

The pass-rate difference is a trend only (65 vs 50, Fisher p = 0.093). The catastrophic failure rate is not: 20 versus 4, odds ratio 5.6, p = 9.7 × 10⁻⁴. Freezing all 36 grafted positions rather than only the 21 contacts makes a design five times more likely to fold into something unrelated to its target backbone. The arms differ in failure mode rather than in central tendency, and the median scRMSD values (0.89 vs 0.82) conceal this entirely.

3. ProteinMPNN score does not predict folding.
Pooled across arms, MPNN score correlates with pLDDT at -0.256, which looks like a weak signal until it is decomposed. The two arms differ in both quantities: fixAll has worse MPNN scores and lower pLDDT. Pooling them manufactures a correlation from that offset.

| arm | MPNN ~ pLDDT | MPNN ~ scRMSD |
|---|---|---|
| fixAll | -0.173 | +0.062 |
| fixContacts | -0.018 | +0.028 |

Within arm, the relationship essentially disappears. Pass rate by MPNN quartile is non-monotonic in both arms (fixAll 0.24/0.44/0.42/0.28; fixContacts 0.49/0.31/0.47/0.53), and chi-square tests are null (p = 0.186 and p = 0.242). The inverse-folding score reports how well a sequence matches a backbone under
ProteinMPNN's own model. It does not predict whether an independent predictor will reproduce that backbone.

4. ESM-2 pseudo-log-likelihood fails against its floor

| sequence | PLL | vs scrambled floor |
|---|---|---|
| designs, fixContacts | −2.071 | +4.9 SD |
| designs, fixAll | −2.237 | +3.5 SD |
| **scrambled (n = 20)** | **−2.677 ± 0.125** | — |
| IL-2, full-length UniProt | −2.568 | +0.9 SD |
| Neo-2/15 (6DG6, gap-free) | −2.757 | **−0.6 SD** |
| IL-2, PDB-derived | −2.834 | **−1.3 SD** |

Neo-2/15 scores inside the shuffled distribution, slightly below its mean. Native human IL-2 scores below it outright. Its full-length sequence recovers to only 0.9 SD above the floor, so splicing artefacts explain part of the deficit, but not the result. Meanwhile the untested designs score 3.5 to 4.9 SD above the floor are far more "natural" than either molecule known to work. The spread among the natural references identifies the mechanism. γc scores -1.497 and IL-2Rα -1.696, both large multi-domain receptor proteins from well-represented families. IL-2 is a short cytokine with comparatively few close homologs. ProteinMPNN produces idealised, helix-rich sequences composed of common residues, which is exactly what a language model finds likely. Therefore, the metric rewards the designs for being more generic than the proteins they are meant to imitate.

5. The apparent PLL-pLDDT correlation is not independent evidence. PLL does correlate with ESMFold confidence (Spearman +0.446, p = 1.5 x 10⁻¹⁵; with scRMSD -0.283, p = 9.7 x 10⁻⁷), and designs that passed score higher than those that failed (-2.069 vs -2.210). This cannot be interpreted as a sequence-only signal predicting structural quality.

6. ESMFold is built on ESM-2. It has different parameter counts, identical family, objective and training corpus. The correlation substantially reflects one model agreeing with its larger sibling, and the two confidence estimates share an inductive bias. The same between-arm confound applies as with MPNN score, and the within-arm values are heterogeneous: fixAll +0.548, fixContacts +0.227.


#### Conclusions

1. 115 of 290 designs are self-consistent, from 20 of 29 backbones. Independent structure prediction returns the intended fold for 40% of sequences. This is the first evidence in the project that the sequences specify their intended fold under an independent predictor. ProteinMPNN and ESMFold are both PDB-trained and both favour idealised geometry, so this is necessary and not sufficient.

2. fixContacts is confirmed as the better arm, on the evidence that actually
discriminates.Stage 3 preferred it on MPNN-score and Cys count. This analysis shows why it matters: a five-fold reduction in catastrophic folding failures (p < 0.001). Freezing structurally intervening residues that bury no receptor surface does not help hold the motif and does make the rest of the protein harder to specify.

3. Two cheap metrics were tested as substitutes for folding and both failed.
ProteinMPNN score carries no within-arm information about foldability. ESM-2 PLL cannot distinguish a validated de novo protein, or native IL-2, from shuffled sequence. Neither was used for selection.

4. The pre-registered three-gate funnel collapsed to one gate. pLDDT > 90 determines 97% of the surviving set. The scRMSD and motif gates were specified on the assumption they would filter independently and they did not.

5. Median scRMSD differs by 0.07 Å between arms. The catastrophic failure count differs five-fold. With a distribution this skewed: median 0.86 Å, maximum 15.5 Å. The informative quantity is the tail, not the centre.

6. A correlation that looked like validation was architectural. The PLL:pLDDT
relationship is real and significant, and it is not independent evidence, because ESMFold's sequence encoder is an ESM-2 model. Two metrics derived from the same model family agreeing with each other is not a cross-check. Identifying this is the reason the PLL result is reported as a failed metric rather than a successful one.

#### Limitaions
Self-consistency is not stability. ESMFold agreement means the sequence specifies the intended fold under one predictor. It does not establish a folding free energy, a melting temperature, expression in any host, or solubility. Stage 5 adds complex prediction. None of it adds thermodynamics.